In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.chat_models.openai import ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.chains.question_answering import load_qa_chain

import os
from dotenv import load_dotenv

In [2]:
load_dotenv()

api_key = os.getenv('OPENAI_API_KEY')
print(api_key[:3])

sk-


In [3]:
# Load modelos (Embeddings e LLM)

embeddings_model = OpenAIEmbeddings()
llm = ChatOpenAI(model='gpt-3.5-turbo', max_tokens=200)

/tmp/ipykernel_21381/1704674422.py:4: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(model='gpt-3.5-turbo', max_tokens=200)


In [4]:
# Carregar o PDF
pdf_link = "./data/visao-estereo-rev.pdf"

loader = PyPDFLoader(pdf_link, extract_images=False)
pages = loader.load_and_split()

In [5]:
# Separar em Chunks (Pedaços de documento)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=20,
    length_function=len,
    add_start_index=True
)

chunks = text_splitter.split_documents(pages)

In [6]:
# Salvar no Vector DB - Chroma

db = Chroma.from_documents(
    chunks,
    embedding=embeddings_model,
    persist_directory="text_index"
)

db.persist()

/tmp/ipykernel_21381/3916169086.py:9: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  db.persist()


In [7]:
# Carregar DB (deixar ele já salvo para evitar retrabalho)
vectordb = Chroma(persist_directory='text_index', embedding_function=embeddings_model)

# Load Retiever
retriever = vectordb.as_retriever(
    search_kwargs={"k":3}
)

# Construção da cadeia de prompt para chamada LLM
chain = load_qa_chain(llm,chain_type="stuff")

/tmp/ipykernel_21381/3782386757.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(persist_directory='text_index', embedding_function=embeddings_model)
/tmp/ipykernel_21381/3782386757.py:10: LangChainDeprecationWarning: This class is deprecated. See the following migration guides for replacements based on `chain_type`:
stuff: https://python.langchain.com/docs/versions/migrating_chains/stuff_docs_chain
map_reduce: https://python.langchain.com/docs/versions/migrating_chains/map_reduce_chain
refine: https://python.langchain.com/docs/versions/migrating_chains/refine_chain
map_rerank: https://python.langchain.com/docs/versions/migrating_chains/map_rerank_docs_chain

See also guides on retrieval and question

In [12]:
def ask(question):
    context = retriever.invoke(question)
    answer = (chain({"input_documents":context, "question":question}, return_only_outputs=True))['output_text']
    return answer,context

In [13]:
user_question = input("User: ")
answer, context = ask(user_question)
print(f"Answer: {answer}")

Answer: A visão estéreo refere-se à capacidade do nosso cérebro de processar as imagens ligeiramente diferentes fornecidas por cada um dos nossos dois olhos para perceber a profundidade e a tridimensionalidade do mundo ao nosso redor. Essa capacidade é fundamental para termos percepção de profundidade e distância.


In [15]:
print(context)

[Document(metadata={'creator': 'Google', 'producer': 'PyPDF', 'page_label': '1', 'creationdate': '', 'source': './data/visao-estereo-rev.pdf', 'start_index': 0, 'page': 0, 'total_pages': 16}, page_content='Visão estéreo\n•Nossos 2 olhos formam imagens \nligeiramente diferentes do mundo\n•A diferença entre as posições de objetos \nnas 2 imagens é chamada de disparidade\n•O termo disparidade pode ser usado com \no significado da discrepância angular na \nposição da imagem de um objeto \nprojetada nos dois olhos\n•O termo distância é a distância física entre \no observador e o objeto,  e o temo \nprofundidade é a distância subjetiva ao \nobjeto que é percebida pelo observador\n•Normalmente, o estudo de stereoscopia é \ndividido em duas partes: primeiro medindo \na disparidade e depois usando-a.\n1'), Document(metadata={'source': './data/visao-estereo-rev.pdf', 'producer': 'PyPDF', 'page_label': '4', 'creationdate': '', 'page': 3, 'total_pages': 16, 'creator': 'Google', 'start_index': 0}, 